In [ ]:
# =============================================================================
# Kaggle房价预测实战 - 完整的机器学习项目流程
# =============================================================================
# 本节演示一个完整的Kaggle竞赛项目，包含：
# - 数据下载与预处理
# - 特征工程（标准化、独热编码）
# - K折交叉验证
# - 模型训练与调参
# - 结果提交

import hashlib
import os
import tarfile
import zipfile
import requests


# =============================================================================
# 数据下载工具函数
# =============================================================================
# DATA_HUB是一个字典，存储了数据集的名称、URL和SHA1校验值
# 这种设计模式便于集中管理多个数据集
DATA_HUB = dict()
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'

def download(name, cache_dir=os.path.join('..', 'data')):  #@save
    """
    下载DATA_HUB中的文件，使用缓存机制避免重复下载
    
    参数:
        name: 数据集名称，必须在DATA_HUB中注册
        cache_dir: 缓存目录，默认在项目上级目录的data文件夹
    
    返回:
        本地文件路径
    
    缓存机制：
        1. 检查文件是否已存在
        2. 使用SHA1校验文件完整性
        3. 校验通过则直接返回，否则重新下载
    """
    assert name in DATA_HUB, f"{name} 不存在于 {DATA_HUB}"
    url, sha1_hash = DATA_HUB[name]
    
    # exist_ok=True: 目录已存在时不报错
    os.makedirs(cache_dir, exist_ok=True)
    
    # 从URL中提取文件名
    fname = os.path.join(cache_dir, url.split('/')[-1])
    
    # 检查缓存文件是否存在且完整
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            # 分块读取大文件，避免内存溢出
            # 每次读取1MB (1048576 bytes)
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        # 校验文件哈希值
        if sha1.hexdigest() == sha1_hash:
            return fname  # 缓存命中
    
    # 缓存未命中，开始下载
    print(f'正在从{url}下载{fname}...')
    # stream=True: 分块下载，适合大文件
    # verify=True: 验证SSL证书
    r = requests.get(url, stream=True, verify=True)
    with open(fname, 'wb') as f:
        f.write(r.content)
    return fname


def download_extract(name, folder=None):  #@save
    """
    下载并解压zip/tar压缩文件
    
    参数:
        name: 数据集名称
        folder: 解压后预期的子文件夹名称（可选）
    """
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, '只有zip/tar文件可以被解压缩'
    
    fp.extractall(base_dir)
    return os.path.join(base_dir, folder) if folder else data_dir


def download_all():  #@save
    """下载DATA_HUB中注册的所有文件"""
    for name in DATA_HUB:
        download(name)


# =============================================================================
# 数据准备与预处理
# =============================================================================
# 如果没有安装pandas，请先安装
# !pip install pandas

%matplotlib inline
import numpy as np
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l

# 在DATA_HUB中注册Kaggle房价数据集
# 格式：(URL, SHA1哈希值)，用于校验文件完整性
DATA_HUB['kaggle_house_train'] = (  #@save
    DATA_URL + 'kaggle_house_pred_train.csv',
    '585e9cc93e70b39160e7921475f9bcd7d31219ce')

DATA_HUB['kaggle_house_test'] = (  #@save
    DATA_URL + 'kaggle_house_pred_test.csv',
    'fa19780a7b011d9b009e8bff8e99922a8ee2eb90')

# 使用pandas加载CSV数据
train_data = pd.read_csv(download('kaggle_house_train'))
test_data = pd.read_csv(download('kaggle_house_test'))

# 合并训练和测试数据，统一进行特征工程
# iloc[:, 1:-1]: 从训练数据中选择第1列到倒数第2列（排除Id和标签）
# iloc[:, 1:]: 从测试数据中选择第1列到最后（排除Id）
all_features = pd.concat((train_data.iloc[:, 1:-1], test_data.iloc[:, 1:]))


# =============================================================================
# 特征工程
# =============================================================================
# 步骤1: 数值特征标准化（Z-score标准化）
# 公式: x' = (x - μ) / σ
# 目的: 使不同尺度的特征具有可比性，加速梯度下降收敛

# 筛选数值型特征（排除字符串/类别型特征）
numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index

# 对数值特征应用标准化
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: (x - x.mean()) / (x.std()))

# 标准化后均值变为0，因此缺失值可以用0填充（相当于用均值填充）
all_features[numeric_features] = all_features[numeric_features].fillna(0)


# 步骤2: 类别特征独热编码（One-Hot Encoding）
# 将类别变量转换为二进制向量，使模型能够处理
# dummy_na=True: 将缺失值也视为一个有效类别
all_features = pd.get_dummies(all_features, dummy_na=True)

# 查看处理后的特征维度
all_features.shape


# =============================================================================
# 数据转换为Tensor
# =============================================================================
# 分离训练集和测试集
n_train = train_data.shape[0]

# 转换为PyTorch张量
train_features = torch.tensor(all_features[:n_train].values, dtype=torch.float32)
test_features = torch.tensor(all_features[n_train:].values, dtype=torch.float32)

# 训练标签：房价，reshape为(n, 1)的列向量
train_labels = torch.tensor(
    train_data.SalePrice.values.reshape(-1, 1), dtype=torch.float32)


# =============================================================================
# 模型定义
# =============================================================================
# 使用均方误差损失函数
loss = nn.MSELoss()

# 输入特征维度
in_features = train_features.shape[1]

def get_net():
    """
    创建简单的线性回归模型
    
    房价预测是回归问题，输出层只有一个神经元
    """
    net = nn.Sequential(nn.Linear(in_features, 1))
    return net


def log_rmse(net, features, labels):
    """
    计算对数RMSE（均方根误差）
    
    使用对数变换的原因：
    1. 房价数据通常呈现长尾分布，对数变换使其更接近正态分布
    2. 更关注相对误差而非绝对误差（高价房的预测误差不应被过度惩罚）
    
    参数:
        net: 神经网络模型
        features: 输入特征
        labels: 真实标签（房价）
    返回:
        RMSE值
    """
    # torch.clamp: 将预测值限制在[1, +∞)范围内
    # 避免log(0)或负数导致的数值问题
    clipped_preds = torch.clamp(net(features), 1, float('inf'))
    
    # 计算对数RMSE
    rmse = torch.sqrt(loss(torch.log(clipped_preds),
                           torch.log(labels)))
    return rmse.item()


def train(net, train_features, train_labels, test_features, test_labels,
          num_epochs, learning_rate, weight_decay, batch_size):
    """
    训练模型
    
    参数:
        net: 待训练的神经网络
        train_features, train_labels: 训练数据
        test_features, test_labels: 验证数据（可为None）
        num_epochs: 训练轮数
        learning_rate: 学习率
        weight_decay: L2正则化系数
        batch_size: 批量大小
    
    返回:
        train_ls, test_ls: 训练和验证的损失历史
    """
    train_ls, test_ls = [], []
    
    # 创建数据迭代器
    train_iter = d2l.load_array((train_features, train_labels), batch_size)
    
    # 使用Adam优化器（适合稀疏特征和大数据集）
    # weight_decay: L2正则化，防止过拟合
    optimizer = torch.optim.Adam(net.parameters(),
                                 lr=learning_rate,
                                 weight_decay=weight_decay)
    
    for epoch in range(num_epochs):
        for X, y in train_iter:
            optimizer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            optimizer.step()
        
        # 记录训练损失
        train_ls.append(log_rmse(net, train_features, train_labels))
        
        # 如果提供了验证数据，记录验证损失
        if test_labels is not None:
            test_ls.append(log_rmse(net, test_features, test_labels))
    
    return train_ls, test_ls


# =============================================================================
# K折交叉验证
# =============================================================================
def get_k_fold_data(k, i, X, y):
    """
    获取K折交叉验证的第i折数据
    
    参数:
        k: 折数（通常取5或10）
        i: 当前折的索引（0到k-1）
        X: 特征数据
        y: 标签数据
    
    返回:
        X_train, y_train: 训练集
        X_valid, y_valid: 验证集（第i折）
    """
    assert k > 1
    fold_size = X.shape[0] // k  # 每折的样本数
    X_train, y_train = None, None
    
    for j in range(k):
        # 计算第j折的切片索引
        idx = slice(j * fold_size, (j + 1) * fold_size)
        X_part, y_part = X[idx, :], y[idx]
        
        if j == i:
            # 第i折作为验证集
            X_valid, y_valid = X_part, y_part
        elif X_train is None:
            # 第一折训练数据
            X_train, y_train = X_part, y_part
        else:
            # 拼接后续折的训练数据
            X_train = torch.cat([X_train, X_part], 0)
            y_train = torch.cat([y_train, y_part], 0)
    
    return X_train, y_train, X_valid, y_valid


def k_fold(k, X_train, y_train, num_epochs, learning_rate, weight_decay,
           batch_size):
    """
    执行K折交叉验证
    
    目的：
    1. 更可靠地评估模型性能
    2. 减少随机划分带来的方差
    3. 检测过拟合
    
    参数:
        k: 折数
        X_train, y_train: 完整训练数据
        其他参数: 传递给train函数
    
    返回:
        平均训练损失和平均验证损失
    """
    train_l_sum, valid_l_sum = 0, 0
    
    for i in range(k):
        # 获取第i折的数据划分
        data = get_k_fold_data(k, i, X_train, y_train)
        
        # 创建新模型（每折使用独立的模型）
        net = get_net()
        
        # 训练模型
        train_ls, valid_ls = train(net, *data, num_epochs, learning_rate,
                                   weight_decay, batch_size)
        
        # 累加最终损失
        train_l_sum += train_ls[-1]
        valid_l_sum += valid_ls[-1]
        
        # 绘制第一折的训练曲线作为示例
        if i == 0:
            d2l.plot(list(range(1, num_epochs + 1)), [train_ls, valid_ls],
                     xlabel='epoch', ylabel='rmse', xlim=[1, num_epochs],
                     legend=['train', 'valid'], yscale='log')
        
        print(f'折{i + 1}，训练log rmse{float(train_ls[-1]):f}, '
              f'验证log rmse{float(valid_ls[-1]):f}')
    
    # 返回平均损失
    return train_l_sum / k, valid_l_sum / k


# =============================================================================
# 模型训练与调参
# =============================================================================
# 超参数设置
k = 5                # 5折交叉验证
num_epochs = 100     # 训练轮数
lr = 5               # 学习率（Adam可以使用较大的初始学习率）
weight_decay = 0     # L2正则化系数（0表示不使用）
batch_size = 64      # 批量大小

# 执行交叉验证
train_l, valid_l = k_fold(k, train_features, train_labels, num_epochs, lr,
                          weight_decay, batch_size)

print(f'{k}-折验证: 平均训练log rmse: {float(train_l):f}, '
      f'平均验证log rmse: {float(valid_l):f}')


# =============================================================================
# 最终训练与提交
# =============================================================================
def train_and_pred(train_features, test_features, train_labels, test_data,
                   num_epochs, lr, weight_decay, batch_size):
    """
    在完整训练数据上训练模型，并生成Kaggle提交文件
    """
    net = get_net()
    
    # 在全部训练数据上训练（无验证集）
    train_ls, _ = train(net, train_features, train_labels, None, None,
                        num_epochs, lr, weight_decay, batch_size)
    
    # 绘制训练曲线
    d2l.plot(np.arange(1, num_epochs + 1), [train_ls], xlabel='epoch',
             ylabel='log rmse', xlim=[1, num_epochs], yscale='log')
    
    print(f'训练log rmse：{float(train_ls[-1]):f}')
    
    # 生成测试集预测
    preds = net(test_features).detach().numpy()
    
    # 将预测结果转换为Kaggle要求的提交格式
    test_data['SalePrice'] = pd.Series(preds.reshape(1, -1)[0])
    submission = pd.concat([test_data['Id'], test_data['SalePrice']], axis=1)
    
    # 保存为CSV文件
    submission.to_csv('submission.csv', index=False)
    print("提交文件已保存为 'submission.csv'")


# 执行最终训练和预测
train_and_pred(train_features, test_features, train_labels, test_data,
               num_epochs, lr, weight_decay, batch_size)